<a href="https://colab.research.google.com/github/CaesarGhazi/Flyrank/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CaesarGhazi/Flyrank/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd, numpy as np, os
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
HF_BASE = "hf://datasets/FlyRank/internship-warehouse"

fact_cols = ["report_date","client_hash_id","content_hash_id","gsc_data_available","ga4_data_available",
             "gsc_impressions","gsc_clicks","gsc_avg_position","ga4_engaged_sessions"]
panel_daily = pd.read_parquet(f"{HF_BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet",
                               columns=fact_cols, storage_options={"token": HF_TOKEN})
dim_content = pd.read_parquet(f"{HF_BASE}/dim_content.parquet",
                               columns=["client_hash_id","content_hash_id","content_type"],
                               storage_options={"token": HF_TOKEN})
for c in ["client_hash_id","content_hash_id"]:
    panel_daily[c] = panel_daily[c].astype("category"); dim_content[c] = dim_content[c].astype("category")
dim_content = dim_content.drop_duplicates(subset=["client_hash_id","content_hash_id"])
panel_daily = panel_daily.merge(dim_content, on=["client_hash_id","content_hash_id"], how="left")

content_level = panel_daily.groupby(["client_hash_id","content_hash_id"], observed=True).agg(
    gsc_impressions=("gsc_impressions","sum"), gsc_clicks=("gsc_clicks","sum"),
    gsc_avg_position=("gsc_avg_position","mean"), ga4_engaged_sessions=("ga4_engaged_sessions","sum"),
    content_type=("content_type","first")).reset_index()
print("Shape:", content_level.shape)

Shape: (331437, 7)


In [2]:
feature_vector = content_level.copy()
feature_vector["ctr"] = feature_vector["gsc_clicks"] / feature_vector["gsc_impressions"].replace(0, np.nan)
feature_vector["engagement_rate"] = feature_vector["ga4_engaged_sessions"] / feature_vector["gsc_clicks"].replace(0, np.nan)
feature_vector["has_position_data"] = feature_vector["gsc_avg_position"].notna().astype(int)

feature_vector = pd.get_dummies(feature_vector, columns=["content_type"], prefix="type", dummy_na=True)

feature_vector["gsc_avg_position_filled"] = feature_vector["gsc_avg_position"].fillna(feature_vector["gsc_avg_position"].median())
feature_vector["ctr_filled"] = feature_vector["ctr"].fillna(0)
feature_vector["engagement_rate_filled"] = feature_vector["engagement_rate"].fillna(0)

print(feature_vector.shape)
feature_vector.head()

(331437, 16)


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_engaged_sessions,ctr,engagement_rate,has_position_data,type_comparison article,type_feedly article,type_keyword article,type_nan,gsc_avg_position_filled,ctr_filled,engagement_rate_filled
0,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,0,0,NaN,0.0,NaN,NaN,0,False,False,True,False,8.505296,0.0,0.0
1,client_0797ff3a1fc9a6a5,content_0236ef736698e17c,0,0,NaN,0.0,NaN,NaN,0,False,False,True,False,8.505296,0.0,0.0
2,client_0797ff3a1fc9a6a5,content_025f6cfd3c298870,0,0,NaN,0.0,NaN,NaN,0,False,False,True,False,8.505296,0.0,0.0
3,client_0797ff3a1fc9a6a5,content_0263d5f9b7a2ecd4,1,0,9.0,0.0,0.0,NaN,1,False,False,True,False,9.000000,0.0,0.0
4,client_0797ff3a1fc9a6a5,content_02752c6c1c60161f,0,0,NaN,0.0,NaN,NaN,0,False,False,True,False,8.505296,0.0,0.0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

gsc_impressions, gsc_clicks — closed, observed same-day counts. No missing values (0 is real).
gsc_avg_position_filled — null when impressions = 0 (confirmed structural), imputed with panel median, flagged via has_position_data. Available same-day for pages with any impressions.
ctr_filled, engagement_rate_filled — null when denominator is 0, filled with 0. Available same-day.
type_* (one-hot content_type) — static attribute, set before the reporting window starts.
has_position_data — binary flag distinguishing genuinely-observed vs. imputed position.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

panel_april = pd.read_parquet(f"{HF_BASE}/fact_content_daily_performance/month=2026-04/data_0.parquet",
                               columns=["client_hash_id","content_hash_id","gsc_clicks"],
                               storage_options={"token": HF_TOKEN})
future_clicks = panel_april.groupby(["client_hash_id","content_hash_id"], observed=True)["gsc_clicks"].sum().rename("future_clicks").reset_index()

leaky = feature_vector.merge(future_clicks, on=["client_hash_id","content_hash_id"], how="left")
leaky["future_clicks"] = leaky["future_clicks"].fillna(0)

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import numpy as np

leak_cols = ["gsc_impressions","gsc_clicks","gsc_avg_position_filled","ga4_engaged_sessions","future_clicks"]
X_leaky = StandardScaler().fit_transform(leaky[leak_cols].fillna(0))
km_leaky = KMeans(n_clusters=2, random_state=42, n_init=10).fit(X_leaky)
rng = np.random.default_rng(42)
idx = rng.choice(len(X_leaky), size=min(20000, len(X_leaky)), replace=False)
print("Silhouette WITH leak:", round(silhouette_score(X_leaky[idx], km_leaky.labels_[idx]), 3))

honest_cols = ["gsc_impressions","gsc_clicks","gsc_avg_position_filled","ga4_engaged_sessions"]
X_honest = StandardScaler().fit_transform(feature_vector[honest_cols].fillna(0))
km_honest = KMeans(n_clusters=2, random_state=42, n_init=10).fit(X_honest)
idx2 = rng.choice(len(X_honest), size=min(20000, len(X_honest)), replace=False)
print("Silhouette WITHOUT leak (honest):", round(silhouette_score(X_honest[idx2], km_honest.labels_[idx2]), 3))

assert "future_clicks" not in feature_vector.columns
print("Leakage check passed: future_clicks absent from final feature set.")

Silhouette WITH leak: 0.93
Silhouette WITHOUT leak (honest): 0.91
Leakage check passed: future_clicks absent from final feature set.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

AI-referral columns — under 0.1% row coverage, too sparse.
is_published, is_deleted, optimization_eligible_date, last_optimized_date — risk of circularity (actions taken in response to performance).
Sealed final-month sample — off-limits categorically.
client_hash_id as a model input — kept as identifier only; using it as a feature risks learning "which client" instead of "what kind of behavior."

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.